# Exercises XP – Week 6, Day 2: Flower Classification using CNN
14 flower species · TensorFlow/Keras. **Run in Google Colab with a GPU** (Runtime → Change runtime type → GPU).
The notebook downloads the course dataset itself.

## Setup: download the dataset and locate the folders

In [ ]:
import os, glob, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

tf.random.set_seed(42)

URL = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%206/W6D3/Flower%20Classification.zip"
if not os.path.exists("flowers"):
    urllib.request.urlretrieve(URL, "flowers.zip")
    with zipfile.ZipFile("flowers.zip") as z:
        z.extractall("flowers")

# Find the training and validation folders: a folder whose sub-folders are the 14 flower classes
def class_dirs(root):
    candidates = []
    for path, dirs, files in os.walk(root):
        if len(dirs) >= 10 and not files:
            candidates.append(path)
    return candidates

candidates = class_dirs("flowers")
print("Folders that contain class sub-folders:", candidates)
TRAIN_DIR = next(c for c in candidates if "train" in c.lower())
VAL_DIR = next((c for c in candidates if "val" in c.lower() or "test" in c.lower()), None)
print("TRAIN_DIR:", TRAIN_DIR, "| VAL_DIR:", VAL_DIR)

## Part 1: Data Exploration and Visualization

In [ ]:
HEIGHT, WIDTH = 64, 64      # reduced from the original resolution to keep training fast
BATCH_SIZE = 64

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.15, subset="training", seed=42,
    image_size=(HEIGHT, WIDTH), batch_size=BATCH_SIZE, label_mode="categorical")
val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.15, subset="validation", seed=42,
    image_size=(HEIGHT, WIDTH), batch_size=BATCH_SIZE, label_mode="categorical")
# The provided validation folder is tiny (98 images), so we use it as the final TEST set
test_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR, image_size=(HEIGHT, WIDTH), batch_size=BATCH_SIZE, label_mode="categorical", shuffle=False)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print(NUM_CLASSES, "classes:", class_names)

In [ ]:
counts = {c: len(glob.glob(os.path.join(TRAIN_DIR, c, "*"))) for c in class_names}
counts = pd.Series(counts).sort_values()
print(counts)
counts.plot(kind="barh", figsize=(8, 5), title="Training images per class")
plt.show()

In [ ]:
def visualize_images(directory, class_name, n=9):
    """Show a 3x3 grid of images for one flower class."""
    files = sorted(glob.glob(os.path.join(directory, class_name, "*")))[:n]
    plt.figure(figsize=(6, 6))
    for i, f in enumerate(files):
        plt.subplot(3, 3, i + 1)
        plt.imshow(plt.imread(f))
        plt.axis("off")
    plt.suptitle(class_name, fontsize=14)
    plt.tight_layout()
    plt.show()

for c in class_names:
    visualize_images(TRAIN_DIR, c)

**Anticipated challenges:**
- **Similar colours and shapes between species:** Black-eyed Susan, Coreopsis, Sunflower and Calendula are all yellow-orange daisy-like flowers; Common Daisy and Water Lily can also look alike.
- **Variation within a species:** roses, tulips and carnations come in many colours, so colour alone isn't reliable.
- **Background and framing:** some photos are close-ups of one flower, others show fields, leaves, pots or people, so the model may learn the background instead of the flower.
- **Low resolution:** shrinking images to 64×64 loses fine details (petal texture, stamens) that distinguish similar species.
- **Unbalanced classes** (see the counts above) can bias the model toward the larger classes.

## Part 2: Model Architecture Design

In [ ]:
def build_cnn(dropout=0.3):
    model = models.Sequential([
        layers.Input(shape=(HEIGHT, WIDTH, 3)),
        layers.Rescaling(1.0 / 255),                  # pixel values 0–255 → 0–1

        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ])
    return model

model = build_cnn()
model.summary()

**Why this architecture:**
- **3 convolution blocks with 32 → 64 → 128 filters:** early layers detect simple patterns (edges, colour patches), deeper layers combine them into petals and flower shapes. Doubling the filters while the image shrinks is a standard, efficient design.
- **3×3 kernels:** small kernels capture local detail with few parameters; stacking them covers larger areas.
- **Max-pooling** after each block halves the image size, which keeps computation low and adds some tolerance to small shifts.
- **Batch Normalization** stabilises and speeds up training.
- **Global Average Pooling** instead of a big Flatten → Dense layer massively reduces parameters, so the model is less likely to overfit.
- **Dropout (0.3)** before the output randomly switches off neurons during training, another guard against overfitting.
- **Softmax output with 14 neurons:** one probability per species.

## Part 3: Hyperparameter Tuning
A small, fast experiment (5 epochs each) comparing optimizers and learning rates, then a full training run with the best setting, early stopping and learning-rate scheduling.

In [ ]:
experiments = [
    ("Adam", 1e-3), ("Adam", 1e-4), ("RMSprop", 1e-3), ("SGD", 1e-2),
]
results = []
for opt_name, lr in experiments:
    m = build_cnn()
    optimizer = {"Adam": tf.keras.optimizers.Adam, "RMSprop": tf.keras.optimizers.RMSprop,
                 "SGD": lambda learning_rate: tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)}[opt_name](learning_rate=lr)
    m.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"])
    h = m.fit(train_ds, validation_data=val_ds, epochs=5, verbose=0)
    results.append({"optimizer": opt_name, "learning_rate": lr, "best_val_accuracy": max(h.history["val_accuracy"])})

experiments_df = pd.DataFrame(results).sort_values("best_val_accuracy", ascending=False)
experiments_df

In [ ]:
best = experiments_df.iloc[0]
print("Best setting:", best["optimizer"], best["learning_rate"])

optimizers = {"Adam": tf.keras.optimizers.Adam, "RMSprop": tf.keras.optimizers.RMSprop,
              "SGD": lambda learning_rate: tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)}

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

model = build_cnn()
model.compile(optimizer=optimizers[best["optimizer"]](learning_rate=best["learning_rate"]),
              loss="categorical_crossentropy", metrics=["accuracy"])
history = model.fit(train_ds, validation_data=val_ds, epochs=25, callbacks=callbacks)

## Part 4: Data Augmentation
Using `ImageDataGenerator` to create randomly modified versions of the training images on the fly.

In [ ]:
augmenter = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=25,
    horizontal_flip=True,
    vertical_flip=False,       # flowers are rarely photographed upside down
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    validation_split=0.15,
)
plain = ImageDataGenerator(rescale=1.0 / 255, validation_split=0.15)

train_aug = augmenter.flow_from_directory(TRAIN_DIR, target_size=(HEIGHT, WIDTH), batch_size=BATCH_SIZE,
                                          subset="training", seed=42)
val_plain = plain.flow_from_directory(TRAIN_DIR, target_size=(HEIGHT, WIDTH), batch_size=BATCH_SIZE,
                                      subset="validation", seed=42)

# Show what augmentation does to one image
batch_images, _ = next(train_aug)
plt.figure(figsize=(8, 3))
for i in range(4):
    plt.subplot(1, 4, i + 1)
    plt.imshow(batch_images[i])
    plt.axis("off")
plt.suptitle("Augmented training images")
plt.show()

In [ ]:
# The generator already rescales, so this model skips the Rescaling layer
def build_cnn_no_rescale(dropout=0.3):
    base = build_cnn(dropout)
    return models.Sequential([layers.Input(shape=(HEIGHT, WIDTH, 3))] + base.layers[1:])

aug_model = build_cnn_no_rescale()
aug_model.compile(optimizer=optimizers[best["optimizer"]](learning_rate=best["learning_rate"]),
                  loss="categorical_crossentropy", metrics=["accuracy"])
aug_history = aug_model.fit(train_aug, validation_data=val_plain, epochs=25, callbacks=callbacks)

print("Best val accuracy without augmentation:", round(max(history.history["val_accuracy"]), 3))
print("Best val accuracy with augmentation:   ", round(max(aug_history.history["val_accuracy"]), 3))

**Which augmentations help most and why:**
- **Horizontal flips, small rotations and zoom** are the most useful: a flower looks the same mirrored, slightly tilted, or photographed closer or further away, so these teach the model real-world variation without changing the species.
- **Small shifts and shears** help the model handle flowers that aren't centred in the frame.
- **Vertical flips** are left out: upside-down flowers (with stems on top) are unrealistic and could confuse the model.
- Augmentation usually makes training accuracy a bit lower but **validation accuracy higher**, because the model can't simply memorise the exact training images. That reduces overfitting.

## Part 5: Performance Evaluation and Analysis

In [ ]:
final_model, final_history = (aug_model, aug_history) \
    if max(aug_history.history["val_accuracy"]) >= max(history.history["val_accuracy"]) else (model, history)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(final_history.history["accuracy"], label="train")
axes[0].plot(final_history.history["val_accuracy"], label="validation")
axes[0].set_title("Accuracy")
axes[0].legend()
axes[1].plot(final_history.history["loss"], label="train")
axes[1].plot(final_history.history["val_loss"], label="validation")
axes[1].set_title("Loss")
axes[1].legend()
plt.show()

In [ ]:
# Evaluate on the held-out test set (the provided validation folder)
needs_rescale = final_model is aug_model
y_true, y_pred, images = [], [], []
for x_batch, y_batch in test_ds:
    x_in = x_batch / 255.0 if needs_rescale else x_batch
    probs = final_model.predict(x_in, verbose=0)
    y_true.extend(np.argmax(y_batch, axis=1))
    y_pred.extend(np.argmax(probs, axis=1))
    images.extend(x_batch.numpy().astype("uint8"))
y_true, y_pred = np.array(y_true), np.array(y_pred)

print(f"Test accuracy: {(y_true == y_pred).mean():.3f}")
print(classification_report(y_true, y_pred, labels=range(NUM_CLASSES), target_names=class_names, zero_division=0))

plt.figure(figsize=(11, 9))
sns.heatmap(confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES)), annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix (test set)")
plt.show()

In [ ]:
# Predictions on test images – red titles are mistakes
plt.figure(figsize=(14, 10))
for i in range(min(16, len(images))):
    plt.subplot(4, 4, i + 1)
    plt.imshow(images[i])
    ok = y_true[i] == y_pred[i]
    plt.title(f"Pred: {class_names[y_pred[i]]}\nTrue: {class_names[y_true[i]]}", color="green" if ok else "red", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

**How to read the results:**
- **Curves:** if training accuracy keeps rising while validation accuracy flattens or drops, the model is **overfitting** (augmentation, dropout and early stopping are meant to limit this). If both stay low, it's **underfitting** (a bigger model, higher resolution or more epochs would help).
- **Per-class metrics:** expect the **yellow daisy-like flowers** (Black-eyed Susan, Coreopsis, Calendula, Sunflower) to be confused with each other most, and distinctive flowers (Iris, Water Lily, Dandelion) to score best. Check the confusion matrix for your exact run.
- **Misclassifications** usually happen when the flower is small in the frame, photographed from an unusual angle, or very similar in colour and shape to another species. The 64×64 resolution makes these cases harder.
- The test set has only **98 images (~7 per class)**, so per-class scores are noisy; one image changes a class's recall by ~14 points.

## Part 6 (optional): Model Saving

In [ ]:
final_model.save("flower_cnn.keras")
print("Model saved as flower_cnn.keras")

reloaded = tf.keras.models.load_model("flower_cnn.keras")
print("Reloaded model works:", reloaded.output_shape)

**Deployment ideas:** wrap the saved model in a small **Flask/FastAPI** web service that accepts a photo and returns the predicted species; deploy it to **Google Cloud Run** or **AWS**; or convert it with **TensorFlow Lite** to run inside a mobile app that identifies flowers from the camera.